## Fleet truck diagnostics notebook

This workbook **materializes the same Python modules** tracked in Git (`utils/`, `retrieval/`, `crewai_layer/`, `app/`, `models/`). Lecture cells used `%%writefile`; the files on disk are the **reference implementation** synced here for clarity.

### What the system does (end-to-end)

1. **Streamlit UI** collects sidebar vehicle facts plus free-text symptoms.
2. **CrewAI crew** (`crew_setup.py`) runs five sequential tasks: structure the complaint → TF-IDF retrieval over the CSV KB → planner tool → safety bulletins → final Markdown synthesis.
3. **Retrieval** (`retrieval/retriever.py`) caches the vector index, applies optional filters, and **rejects statistically weak cosine matches** so the LLM cannot "shop" random historical trucks.
4. **Tasks** inject `{user_message}`, `{conversation_history}`, and `{ui_context}` into prompts so every agent sees the same operator truth.

### Colab vs local

| Where | What to do |
|-------|------------|
| **Google Colab** | Mount Drive, `%cd` into this project folder, run cells top-to-bottom, then use the Streamlit tunnel cell (Colab-specific). |
| **Local clone** | Install deps per `requirements.txt`, put the Kaggle CSV at `data/logistics_vehicle_maintenance_history.csv`, then `python -m streamlit run app/streamlit_app.py` from the repo root. You can skip Drive cells. |

### Cell tour (read the `#` banners inside each code cell)

- **Environment & deps** — API keys, pip installs, warning filters.
- **Scaffold / tiny data** — create folders plus `safety_bulletins.csv`.
- **Library modules** — every `%%writefile` block mirrors one production file (`README.md` documents env tuning such as `TRUCK_KB_MIN_TFIDF_SIM`).
- **Optional probes** — load the big CSV sanity check when you finish exports.
- **Streamlit launcher** — Colab tunnels; locals should prefer the README command.

_No step should be mysterious: each code section starts with comments describing inputs, artifacts on disk, and downstream consumers._



In [ ]:
# --- Section 1: Drive mount + secrets (Colab lecture path) ---
# Mount Drive, cd into YOUR copy of TruckDiagnosisTool, then hydrate OPENAI_* via Colab Secrets.
# Locally: SKIP this entire cell and load keys from '.streamlit/secrets.toml' or your shell instead.

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Working directory → same folder you'd `git clone` locally
%cd /content/drive/MyDrive/LLMs_MGT802/FinalProject

import os
from google.colab import userdata

# Persist OpenAI secrets from Colab Secrets / KV store.
# Accept either class-project secret name; do not paste the raw API key into the notebook.
os.environ['OPENAI_API_KEY'] = userdata.get('openai_api_key2') or userdata.get('openai_api_key')
os.environ["OPENAI_MODEL_NAME"] = 'gpt-4o-mini'


In [ ]:
# --- Dependencies: CrewAI + Streamlit + pandas/scikit-learn ---
%%capture
!pip install --upgrade crewai crewai-tools qdrant-client==1.15.1 langchain_community \
    streamlit pandas scikit-learn python-dotenv


In [ ]:
# --- Silence noisy library warnings during iterative demos ---
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# --- Section 2: Scaffold folders that %%writefile cells will populate ---
# Lecture path below targets Google Drive; on your laptop, point base_dir at Path.cwd() instead.

import os

base_dir = "/content/drive/MyDrive/LLMs_MGT802/FinalProject"

dirs = [
    "app",
    "crewai_layer",
    "retrieval",
    "models",
    "utils",
    "tests",
    "data",
]

for d in dirs:
    full = os.path.join(base_dir, d)
    os.makedirs(full, exist_ok=True)
    init_path = os.path.join(full, "__init__.py")
    if not os.path.exists(init_path):
        with open(init_path, "w") as f:
            f.write("# Package init\n")

print("Directory structure created.")


In [ ]:
import os
os.makedirs("data", exist_ok=True)

In [ ]:
# --- Generated asset: data/safety_bulletins.csv ---
# Purpose: Hand-authored subsystem hazards merged into the SAFETY agent output.

%%writefile data/safety_bulletins.csv
subsystem,hazard,note
Cooling,Burns from hot coolant,"Never remove radiator or expansion tank cap when the engine is hot. Allow sufficient cool-down time."
Turbo,Injury from rotating components,"Do not work near the turbocharger or fan while the engine is running. Keep loose clothing and tools clear."
Fuel,Fire and environmental risk,"Work in a well-ventilated area away from ignition sources. Use spill trays and absorbent material for fuel leaks."
Electrical,Shock and short-circuit,"Disconnect batteries before servicing main power cables. Use insulated tools when possible."
General,Vehicle movement,"Always secure the truck with parking brake and wheel chocks before working underneath or around moving components."


In [ ]:
import os
os.makedirs("utils", exist_ok=True)

In [ ]:
# --- Exported module: utils/config.py ---
# Purpose: Canonical paths for the KB CSV / safety bulletin file + CrewAI LLM factories.
# Tuning knobs: OPENAI_MODEL_NAME, OPENAI_PLANNER_MODEL_NAME, OPENAI_MAIN_TEMP, OPENAI_PLANNER_TEMP.

%%writefile utils/config.py
import os
import tomllib
from dataclasses import dataclass
from pathlib import Path

from crewai import LLM

BASE_DIR = Path(__file__).resolve().parent.parent

DATA_DIR = BASE_DIR / "data"

TRUCK_KB_PATH = DATA_DIR / "logistics_vehicle_maintenance_history.csv"

SAFETY_BULLETINS_PATH = DATA_DIR / "safety_bulletins.csv"

SECRET_KEY_NAMES = ("OPENAI_API_KEY", "OPENAI_API_KEY2", "openai_api_key2", "openai_api_key")


def _key_from_streamlit_secrets_file() -> str | None:
    secrets_path = BASE_DIR / ".streamlit" / "secrets.toml"
    if not secrets_path.exists():
        return None

    try:
        data = tomllib.loads(secrets_path.read_text(encoding="utf-8-sig"))
    except Exception:
        return None

    for candidate in SECRET_KEY_NAMES:
        value = data.get(candidate)
        if value:
            return str(value).strip()

    return None


def ensure_openai_api_key() -> str | None:
    """
    Normalize project-specific key names to the environment variable expected by CrewAI.
    """
    if value := os.getenv("OPENAI_API_KEY"):
        return value

    for candidate in SECRET_KEY_NAMES[1:]:
        value = os.getenv(candidate)
        if value:
            os.environ["OPENAI_API_KEY"] = value
            return value

    if value := _key_from_streamlit_secrets_file():
        os.environ["OPENAI_API_KEY"] = value
        return value

    return None


@dataclass
class ModelConfig:
    main_model: str = os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini")
    planner_model: str = os.getenv("OPENAI_PLANNER_MODEL_NAME", "gpt-4o-mini")


def get_main_llm() -> LLM:
    api_key = ensure_openai_api_key()
    cfg = ModelConfig()
    return LLM(model=cfg.main_model, temperature=float(os.getenv("OPENAI_MAIN_TEMP", "0.4")), api_key=api_key)


def get_planner_llm() -> LLM:
    api_key = ensure_openai_api_key()
    cfg = ModelConfig()
    return LLM(model=cfg.planner_model, temperature=float(os.getenv("OPENAI_PLANNER_TEMP", "0.2")), api_key=api_key)


In [ ]:
# --- Exported module: utils/logging_utils.py ---
# Purpose: Thin logging helper used by crew_setup when kicking off sequential tasks.

%%writefile utils/logging_utils.py
import logging
from typing import Optional


def get_logger(name: str = "truck_diag") -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        logger.setLevel(logging.INFO)
        ch = logging.StreamHandler()
        fmt = logging.Formatter(
            "[%(asctime)s] [%(levelname)s] %(name)s - %(message)s",
            datefmt="%Y-%m-%d %H:%M:%S",
        )
        ch.setFormatter(fmt)
        logger.addHandler(ch)
    return logger


_logger: Optional[logging.Logger] = None


def get_project_logger() -> logging.Logger:
    global _logger
    if _logger is None:
        _logger = get_logger()
    return _logger


In [ ]:
# --- Exported module: utils/security.py ---
# Purpose: Naive substring guard against obvious prompt-abuse payloads before CrewAI runs.

%%writefile utils/security.py
from typing import Tuple


BLOCKED_KEYWORDS = [
    "override system prompt",
    "ignore previous instructions",
    "reveal your system prompt",
    "exfiltrate api key",
    "self-harm",
    "suicide",
]


def sanitize_user_input(text: str) -> Tuple[bool, str]:
    lowered = text.lower()
    for kw in BLOCKED_KEYWORDS:
        if kw in lowered:
            return (
                False,
                "For safety reasons I can't follow that instruction. "
                "Please describe the truck issue, symptoms, and context instead.",
            )
    return True, text.strip()


In [ ]:
import os
os.makedirs("retrieval", exist_ok=True)

In [ ]:
# --- Exported module: retrieval/kb_loader.py ---
# Purpose: Load and cache pandas DataFrames (issues CSV + bulletin CSV keyed by subsystem).

%%writefile retrieval/kb_loader.py
import os
from pathlib import Path
from typing import Optional

import pandas as pd

from utils.config import SAFETY_BULLETINS_PATH, TRUCK_KB_PATH

_ISSUES_CACHE_PATH: Optional[str] = None
_ISSUES_DF = None


def load_truck_issues(path: Optional[Path] = None) -> pd.DataFrame:
    """
    Load the logistics vehicle maintenance CSV. Cached in-memory by path+mtime so
    Streamlit reruns don't re-parse tens of thousands of rows every interaction.
    """
    global _ISSUES_CACHE_PATH, _ISSUES_DF

    resolved = Path(path or TRUCK_KB_PATH)
    try:

        key = f"{resolved.resolve()}::{os.path.getmtime(resolved)}"

    except OSError:

        key = str(resolved)

    if _ISSUES_DF is not None and _ISSUES_CACHE_PATH == key:

        return _ISSUES_DF

    _ISSUES_CACHE_PATH = key
    _ISSUES_DF = pd.read_csv(resolved)
    return _ISSUES_DF


def truck_kb_mtime_key() -> str:
    resolved = Path(TRUCK_KB_PATH)
    try:

        return f"{resolved.resolve()}:{os.path.getmtime(resolved)}"

    except OSError:

        return str(resolved)


def invalidate_truck_issues_cache() -> None:

    global _ISSUES_CACHE_PATH, _ISSUES_DF

    _ISSUES_CACHE_PATH = None

    _ISSUES_DF = None

    try:

        from retrieval import retriever as _retriever

        _retriever.clear_retriever_cache()

    except ImportError:

        pass


def load_safety_bulletins(path: Optional[Path] = None) -> pd.DataFrame:
    resolved = Path(path or SAFETY_BULLETINS_PATH)

    df = pd.read_csv(resolved)

    df["subsystem"] = df["subsystem"].astype(str).str.strip().str.lower()

    return df


In [ ]:
# --- Exported module: retrieval/retriever.py ---
# Purpose: TF-IDF cosine retrieval with global index caching + similarity gating.

%%writefile retrieval/retriever.py
"""TF-IDF retrieval over the truck maintenance KB with caching and similarity gating."""

from __future__ import annotations

import os
import re
from threading import Lock
from typing import Any

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

from retrieval.kb_loader import load_truck_issues, load_safety_bulletins, truck_kb_mtime_key

# Lowest max cosine similarity to accept retrieved rows as "related" (tune via env).
_MIN_TOP_SIM = float(os.getenv("TRUCK_KB_MIN_TFIDF_SIM", "0.06"))
# Margin between top-1 and top-2; tiny margin => ambiguous / bogus match.
_MIN_MARGIN = float(os.getenv("TRUCK_KB_MIN_TFIDF_MARGIN", "0.005"))

_CACHE_LOCK = Lock()
_ISSUES_INDEX_CACHE: dict[str, Any] = {}


def clear_retriever_cache() -> None:
    """Drop in-process TF-IDF index (useful after swapping CSV files)."""
    global _ISSUES_INDEX_CACHE
    with _CACHE_LOCK:
        _ISSUES_INDEX_CACHE.clear()


def prime_truck_issues_index() -> None:
    """Warm the KB index once (call from Streamlit @st.cache_resource)."""
    _build_issues_search_index(force=False)


def _electrical_patterns(text_lower: str) -> bool:
    """Avoid substring 'ground' matching LLM jargon like 'grounding in answers'."""
    electrical_tokens = (
        r"\bbattery\b",
        r"\bstarter\b",
        r"\balternator\b",
        r"\bvoltage\b",
        r"\b12v\b",
        r"\belectrical\b",
        r"ground\s*strap",
        r"chassis\s*ground",
        r"negative\s*cable",
        r"positive\s*cable",
        r"\bloom\b.*\bwiring\b",
        r"\becm\b",
    )
    return any(re.search(p, text_lower) for p in electrical_tokens)


def classify_subsystem(symptom_text: str) -> str:
    text = symptom_text.lower()
    if any(k in text for k in ("overheat", "coolant", "radiator", "fan", "temperature")):
        return "cooling"
    if any(k in text for k in ("turbo", "boost", "whistle")):
        return "turbo"
    if any(k in text for k in ("fuel", "injector", "rail", "def fluid", "adblue")):
        return "fuel"
    # "filter" alone is ambiguous; tie to intake/air when obvious
    if "air filter" in text or "fuel filter" in text:
        return "fuel" if "fuel filter" in text else "general"
    if _electrical_patterns(text):
        return "electrical"
    return "general"


def _find_make_model_column(cols: list[str]) -> str | None:
    for c in cols:
        cl = c.lower()
        if "make_and_model" in cl or ("make" in cl and "model" in cl):
            return c
    return None


def _find_subsystem_like_column(cols: list[str]) -> str | None:
    for c in cols:
        cl = c.lower()
        if "subsystem" in cl:
            return c
        if cl in {"system", "component"} or "system" == cl:
            return c
        if "component" in cl:
            return c
    return None


def _build_issues_search_index(force: bool) -> tuple[Any, TfidfVectorizer, Any, list[str]]:
    """
    Cached: full dataframe reference (not copied), sparse TF-IDF matrix, text column list.
    """
    global _ISSUES_INDEX_CACHE

    with _CACHE_LOCK:

        df = load_truck_issues()

        cols = df.columns.tolist()

        keysig = "::".join(cols[: min(40, len(cols))])

        sig = keysig + "::" + str(df.shape)

        key = f"{truck_kb_mtime_key()}:{hash(sig)}"

        if not force and key == _ISSUES_INDEX_CACHE.get("key"):

            b = _ISSUES_INDEX_CACHE

            return b["df"], b["vectorizer"], b["matrix"], b["text_cols"]

        text_cols = df.select_dtypes(include=["object"]).columns.tolist()

        text_cols = [c for c in text_cols if c.lower() != "vehicle_id"]

        blob = df[text_cols].fillna("").agg(" ".join, axis=1) if text_cols else None

        if blob is None:

            corpus = []

        else:

            corpus = blob.astype(str).tolist()

        vectorizer = TfidfVectorizer(
            stop_words="english",
            max_features=int(os.getenv("TRUCK_KB_TFIDF_MAX_FEATURES", "20000")),
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
        )

        matrix = vectorizer.fit_transform(corpus)

        bundle = {
            "key": key,
            "df": df,
            "vectorizer": vectorizer,
            "matrix": matrix,
            "text_cols": text_cols or [],
            "blob": blob,
        }

        _ISSUES_INDEX_CACHE = bundle

        return df, vectorizer, matrix, text_cols or []


def retrieve_issues(context: dict[str, Any], top_k: int = 3) -> list[dict[str, Any]]:
    """
    Return top_k maintenance rows grounded in cosine similarity against the symptom query.

    - Does NOT widen to the entire 90k KB when structured filters wipe the frame
      (that was the main hallucination trigger).
    - Returns [] when the match is statistically weak vs the query — upstream tools
      should treat this as NO_MATCHES and avoid inventing plausible truck stories from
      unrelated rows.
    """

    df, vectorizer, matrix, text_cols = _build_issues_search_index(force=False)

    make = str(context.get("make", "") or "").strip()
    model = str(context.get("model", "") or "").strip()
    subsystem = str(context.get("subsystem", "") or "").strip().lower()

    symptoms = str(
        context.get("symptoms", "")
        or context.get("primary_symptoms", "")
        or context.get("free_text", "")
        or ""
    ).strip()

    ui_bits = []
    if context.get("mileage"):
        ui_bits.append(f"mileage: {context.get('mileage')}")
    if context.get("year"):
        ui_bits.append(f"year: {context.get('year')}")
    ui_tail = "; ".join(ui_bits)

    if not subsystem and symptoms:
        subsystem = classify_subsystem(symptoms)
        context["subsystem"] = subsystem

    query = symptoms or ""
    if not query.strip():
        if make or model:
            query = " ".join(x for x in (make, model) if x)
        if ui_tail:
            query = (query + " " + ui_tail).strip() if query else ui_tail

    if not query:
        context["retrieval_note"] = "No symptom/query text provided; skipping KB retrieval."
        return []

    cols = df.columns.tolist()
    mm_col = _find_make_model_column(cols)
    sub_col = _find_subsystem_like_column(cols)

    n_docs = matrix.shape[0]
    eligible = np.ones(n_docs, dtype=bool)

    if mm_col and (make or model):
        s = df[mm_col].astype(str).str.strip()
        if make and model:
            mk_l, md_l = make.lower(), model.lower()

            lowered = s.str.lower()

            mm_mask = lowered.str.contains(re.escape(mk_l), regex=True, na=False) & lowered.str.contains(
                re.escape(md_l),
                regex=True,
                na=False,
            )

        elif make:

            mm_mask = s.str.lower().str.contains(re.escape(make.lower()), na=False)

        else:

            mm_mask = s.str.lower().str.contains(re.escape(model.lower()), na=False)

        subset_count = int(mm_mask.sum())

        eligible &= mm_mask.to_numpy(dtype=bool)
        context["kb_make_model_candidates"] = subset_count

        if subset_count == 0:
            context["retrieval_note"] = (
                f"No rows match make/model `{make}` `{model}`; searching full KB "
                "by symptom similarity (no fabricated vehicle linkage)."
            )
            eligible[:] = True

    elif make or model:
        context["retrieval_note"] = "Dataset has no make/model column detected; symptom-only retrieval."

    if sub_col is not None and subsystem and subsystem != "general":
        ssub = df[sub_col].astype(str).str.lower()
        subsystem_mask = ssub.str.contains(re.escape(subsystem), na=False)
        tentative = eligible & subsystem_mask.to_numpy(dtype=bool)

        # Subsystem heuristic can be brittle on synthetic schemas — avoid emptying eligibility.
        if int(tentative.sum()) >= 20:
            eligible = tentative

    q_vec = vectorizer.transform([query])
    cos = q_vec.dot(matrix.transpose())
    dense = cos.toarray().ravel()

    dense_masked = dense.copy()

    masked_out_count = int((~eligible).sum())

    dense_masked[~eligible] = -1.0

    if masked_out_count == n_docs:
        dense_masked = dense.copy()

    ranked = np.argsort(dense_masked)[::-1][: max(top_k * 8, top_k)]

    vals = dense_masked[ranked]
    best_idx: list[int] = []

    for ix, idx in enumerate(ranked):

        sc = vals[ix]

        if sc <= -1e-6:
            continue

        best_idx.append(int(idx))

        if len(best_idx) >= top_k * 3:
            break

    scores = dense_masked[np.array(best_idx, dtype=np.int64)] if best_idx else np.array([])

    max_sim = float(scores.max()) if scores.size else 0.0
    sorted_scores = np.sort(scores)[::-1]
    second = float(sorted_scores[1]) if sorted_scores.size > 1 else 0.0
    margin = max_sim - second

    context["retrieval_best_similarity"] = max_sim

    gate_fail = scores.size == 0 or max_sim < _MIN_TOP_SIM or (
        margin < _MIN_MARGIN and max_sim < max(_MIN_TOP_SIM * 1.5, _MIN_TOP_SIM + 0.02)
    )

    if gate_fail:
        context["retrieval_note"] = (
            (context.get("retrieval_note") or "")
            + f" Retrieval skipped weak matches (top_sim={max_sim:.4f})."
        ).strip()

        context["weak_kb_match"] = True
        return []

    sorted_idx = sorted(zip(scores.tolist(), best_idx), reverse=True)[:top_k]

    out: list[dict[str, Any]] = []
    for sc, ri in sorted_idx:

        rowdict = df.iloc[int(ri)].to_dict()

        rowdict["_retrieval_similarity"] = float(sc)

        out.append(rowdict)

    context["retrieval_similarity_margin"] = margin

    return out


def retrieve_safety_notes(subsystem: str | None) -> list[dict[str, str]]:
    df = load_safety_bulletins()
    subsystem_clean = (subsystem or "").strip().lower()
    if subsystem_clean:

        df_sub = df[df["subsystem"] == subsystem_clean]

        if df_sub.empty:

            df_sub = df[df["subsystem"].isin(["general"])]

    else:

        df_sub = df[df["subsystem"].isin(["general"])]

    return df_sub.to_dict(orient="records")


In [ ]:
import os
os.makedirs("models", exist_ok=True)

In [ ]:
# --- Exported module: models/llm_client.py ---
# Purpose: Bridge between Fleet agents (supervisor vs planner personas) and config.py LLMs.

%%writefile models/llm_client.py
from crewai import LLM

from utils.config import get_main_llm, get_planner_llm


def get_supervisor_llm() -> LLM:
    return get_main_llm()


def get_planner_safety_llm() -> LLM:
    return get_planner_llm()


In [ ]:
import os
os.makedirs("crewai_layer", exist_ok=True)

In [ ]:
# --- Exported module: crewai_layer/tools.py ---
# Purpose: CrewAI @tool wrappers around retrieval plus diagnostic draft + safety overlays.

%%writefile crewai_layer/tools.py
from typing import Any

from crewai.tools import tool

from retrieval.retriever import retrieve_issues, retrieve_safety_notes


@tool("retrieve_issue_info_tool")
def retrieve_issue_info_tool(query_context: dict[str, Any]) -> str:
    """
    Tool: Retrieve known vehicle maintenance rows from the fleet maintenance CSV.

    Pass keys like: make, model, subsystem, symptoms/primary_symptoms/free_text.

    Tool output is verbatim row fields plus an internal similarity score column
    `_retrieval_similarity` when matches are statistically strong enough.
    """
    issues = retrieve_issues(dict(query_context or {}), top_k=5)
    if not issues:

        return "NO_MATCHES_FOUND"

    lines: list[str] = ["CANDIDATE_ISSUES_START"]
    for i, issue in enumerate(issues, start=1):

        lines.append(f"- ISSUE #{i}")

        keys = sorted(issue.keys())

        for key in keys:

            val = issue[key]

            lines.append(f"  {key}: {val}")

    lines.append(
        "\nRULE: Probable Causes must explicitly tie KB rows back to the operator's wording; "
        "discard any row whose vehicle/details conflict with STRUCTURED CONTEXT."
    )
    lines.append("CANDIDATE_ISSUES_END")
    return "\n".join(lines)


@tool("build_diagnostic_plan_tool")
def build_diagnostic_plan_tool(candidate_issues_block: str, context: dict[str, Any]) -> str:
    """
    Take candidate issues plus UI context dict and produce draft planning scaffolding.
    """
    return (
        "DIAGNOSTIC_PLAN_DRAFT_START\n"
        "Treat the retrieved maintenance rows as SOFT evidence anchors for {user_message}.\n"
        "If candidate block is NO_MATCHES_FOUND or rows clearly mismatch the customer's truck/\n"
        "symptoms, write a GENERAL diagnostic playbook only (verification steps first), cite NO\n"
        "specific phantom vehicle from the KB.\n"
        f"{candidate_issues_block}\n\n"
        "Context from UI/forms + structured summary:\n"
        f"{context}\n"
        "DIAGNOSTIC_PLAN_DRAFT_END"
    )


@tool("safety_check_tool")
def safety_check_tool(diagnostic_plan_text: str, subsystem: str | None = None) -> str:
    """
    Attach safety bulletins to a drafted plan.
    """
    notes = retrieve_safety_notes(subsystem)
    lines = ["SAFETY_REVIEW_START", "Original plan:", diagnostic_plan_text, "\nRelevant safety bulletins:"]
    for n in notes:

        lines.append(f"- [{n['subsystem']}] {n['hazard']}: {n['note']}")

    lines.append("SAFETY_REVIEW_END")
    return "\n".join(lines)


In [ ]:
# --- Exported module: crewai_layer/agents.py ---
# Purpose: Four specialist personas (Supervisor / Retrieval / Planner / Safety Review).

%%writefile crewai_layer/agents.py
from typing import List

from crewai import Agent

from crewai_layer.tools import (
    build_diagnostic_plan_tool,
    retrieve_issue_info_tool,
    safety_check_tool,
)
from models.llm_client import get_planner_safety_llm, get_supervisor_llm


def create_supervisor_agent() -> Agent:
    return Agent(
        role="Truck Diagnostic Supervisor Agent",
        goal=(
            "Coordinate truck diagnostics safely; final answers MUST reflect ONLY the user's stated "
            "vehicle + symptoms ({user_message}) and sidebar fields ({ui_context})."
        ),
        backstory=(
            "Workshop foreman: structured diagnostics, refusal to retrofit unrelated KB rows into "
            "the customer's story, and concise Markdown output."
        ),
        llm=get_supervisor_llm(),
        verbose=False,
        allow_delegation=False,
        tools=[],
    )


def create_retriever_agent() -> Agent:
    return Agent(
        role="Truck Diagnostic Retrieval Agent",
        goal=(
            "Call retrieve_issue_info_tool exactly once using symptoms + STRUCTURED CONTEXT from "
            "Task 1 merged with {ui_context}; never hallucinate unseen rows."
        ),
        backstory=(
            "You only summarize tool output verbatim; tool already enforces similarity gating."
        ),
        llm=get_planner_safety_llm(),
        verbose=False,
        allow_delegation=False,
        tools=[retrieve_issue_info_tool],
    )


def create_planner_agent() -> Agent:
    return Agent(
        role="Diagnostic Planner Agent",
        goal=(
            "Turn KB candidates + customer's words into actionable steps WITHOUT inventing mismatched trucks."
        ),
        backstory="Senior diagnostics engineer disciplined about evidence-vs-speculation labeling.",
        llm=get_planner_safety_llm(),
        verbose=False,
        allow_delegation=False,
        tools=[build_diagnostic_plan_tool],
    )


def create_safety_agent() -> Agent:
    return Agent(
        role="Safety Review Agent",
        goal="Layer PPE/Lock-out guidance onto the drafted plan strictly from safety bullets + plan text.",
        backstory=(
            "Fleet safety officer insisting on verbatim hazard language when uncertain and blocking "
            "reckless troubleshooting steps."
        ),
        llm=get_planner_safety_llm(),
        verbose=False,
        allow_delegation=False,
        tools=[safety_check_tool],
    )


def create_all_agents() -> List[Agent]:
    return [
        create_supervisor_agent(),
        create_retriever_agent(),
        create_planner_agent(),
        create_safety_agent(),
    ]


In [ ]:
# --- Exported module: crewai_layer/tasks.py ---
# Purpose: Task prompts with explicit template variables for user + UI context wiring.

%%writefile crewai_layer/tasks.py
from crewai import Agent, Task


def create_analyze_symptom_task(supervisor: Agent) -> Task:
    return Task(
        description=(
            "CONVERSATION (last turns):\n{conversation_history}\n\n"
            "LATEST USER MESSAGE:\n{user_message}\n\n"
            "SIDEBAR / FORM CONTEXT (truth for vehicle meta):\n{ui_context}\n\n"
            "Extract BOTH the user's verbatim complaint AND sidebar vehicle facts.\n"
            "Return ONE short paragraph recap plus a STRUCTURED_JSON block formatted EXACTLY as:\n"
            "STRUCTURED_CONTEXT_START\n"
            '{"vehicle_make":"","vehicle_model":"","year":"","mileage":"","load_condition":"","'
            'ambient_temperature":"","recent_maintenance":"","primary_symptoms":"","operating_conditions":"",'
            '"suspected_subsystem":"","maintenance_history_signals":"","free_text":"<full user wording>"}'
            "\nSTRUCTURED_CONTEXT_END\n"
            "If information is absent, leave empty strings—do NOT guess vehicle identity."
        ),
        expected_output="Paragraph + fenced STRUCTURED_CONTEXT JSON block usable by retrieval tool.",
        agent=supervisor,
    )


def create_retrieve_issues_task(retriever: Agent) -> Task:
    return Task(
        description=(
            "Structured context sits in outputs of Task 1; DO NOT overwrite it.\n\n"
            "TASK: Immediately call retrieve_issue_info_tool once with Python dict literal keys matching "
            '{"make":"vehicle_make","model":"vehicle_model","symptoms":"<primary_symptoms + free_text + '
            '{user_message} keywords>","subsystem":"suspected_subsystem","mileage":"<from JSON>",'
            '"year":"<from JSON>"}'
            "; inject missing blanks if necessary.\n"
            "Return ONLY the verbatim tool blob (CANDIDATE_ISSUES_*) plus a ONE-line note PASS/FAIL on "
            "whether rows appear relevant versus {user_message}."
        ),
        expected_output="Tool output verbatim + one-line PASS/FAIL relevance note.",
        agent=retriever,
        tools=retriever.tools,
    )


def create_plan_diagnostics_task(planner: Agent) -> Task:
    return Task(
        description=(
            "Inputs: Structured context from Task 1, retrieval blob from Task 2, ORIGINAL SYMPTOMS {user_message}.\n\n"
            "CALL build_diagnostic_plan_tool(candidate_issues_block=<Task2 output>, context=<parsed JSON dict from "
            "Task1 STRUCTURED CONTEXT>, ... ) EXACT signatures per tool docs.\n"
            "Then polish into DIAGNOSTIC_PLAN_REFINED_START / END emphasizing:\n"
            "- Separate **Customer vehicle facts** vs **Historical KB rows**\n"
            "- If retrieval was NO_MATCHES or FAIL, give generic verification-first workflow & stop inventing fleets."
        ),
        expected_output="DIAGNOSTIC_PLAN draft wrapper + DIAGNOSTIC_PLAN_REFINED block.",
        agent=planner,
        tools=planner.tools,
    )


def create_review_safety_task(safety_agent: Agent) -> Task:
    return Task(
        description=(
            "Given refined diagnostic plan from Task 3 and suspected_subsystem from structured JSON, invoke "
            "safety_check_tool with that subsystem string plus the plan body.\n"
            "Preserve SAFETY markers from tool.\n\n"
            "Customer symptoms reference: {user_message}"
        ),
        expected_output="SAFETY_REVIEW block referencing original plan bullets.",
        agent=safety_agent,
        tools=safety_agent.tools,
    )


def create_synthesize_response_task(supervisor: Agent) -> Task:
    return Task(
        description=(
            "Supervisor FINAL answer for {user_message} using Tasks 1-4 ONLY.\n\n"
            "**GROUNDING CONTRACT**\n"
            "- Summary mirrors sidebar + STRUCTURED CONTEXT + user wording; NEVER swap in KB vehicle IDs.\n"
            "- Probable Causes: if retrieval weak/empty, headline that KB lacked matches and speak hypothetically.\n"
            "- When citing KB rows, start bullets with similarity score `_retrieval_similarity` echo if present "
            "or label as 'Possible parallel case:'.\n"
            "- Forbidden: pretending a Volvo case is the user's Peterbilt (example pattern).\n\n"
            "## Summary of the Situation\n"
            "## Evidence from Fleet KB (explicitly labeled, optional)\n"
            "## Probable Causes\n"
            "## Diagnostic Workflow (Step-by-Step)\n"
            "## Required Tools & Parts\n"
            "## Estimated Time\n"
            "## Safety Considerations\n"
            "## Notes & Uncertainties\n"
            "Close with reassurance to verify on-vehicle readings/codes.\n\n"
            "Operational context recap: sidebar JSON {ui_context}\n"
            "Conversation tail: {conversation_history}"
        ),
        expected_output="Markdown adhering to headings above; truthful about retrieval strength.",
        agent=supervisor,
    )


In [ ]:
# --- Exported module: crewai_layer/crew_setup.py ---
# Purpose: Sequential Crew wiring + conversation compaction + primes TF-IDF cache.

%%writefile crewai_layer/crew_setup.py
from __future__ import annotations

import json
from typing import Any, Dict, List

from crewai import Crew, Process

from crewai_layer.agents import create_all_agents
from crewai_layer.tasks import (
    create_analyze_symptom_task,
    create_plan_diagnostics_task,
    create_retrieve_issues_task,
    create_review_safety_task,
    create_synthesize_response_task,
)
from retrieval.retriever import prime_truck_issues_index
from utils.logging_utils import get_project_logger

logger = get_project_logger()


def build_tasks(agents) -> List:
    supervisor = [a for a in agents if "Supervisor" in a.role][0]
    retriever = [a for a in agents if "Retrieval" in a.role][0]
    planner = [a for a in agents if "Planner" in a.role][0]
    safety = [a for a in agents if "Safety" in a.role][0]

    t1 = create_analyze_symptom_task(supervisor)
    t2 = create_retrieve_issues_task(retriever)
    t3 = create_plan_diagnostics_task(planner)
    t4 = create_review_safety_task(safety)
    t5 = create_synthesize_response_task(supervisor)

    t2.context = [t1]
    t3.context = [t1, t2]
    t4.context = [t1, t3]
    t5.context = [t1, t2, t3, t4]

    return [t1, t2, t3, t4, t5]


def _format_conversation(history: List[Dict[str, str]], max_turns: int = 14, max_chars: int = 6000) -> str:
    snippets: List[str] = []

    tail = history[-max_turns:] if history else []

    for turn in tail:
        role = str(turn.get("role", "user"))
        msg = str(turn.get("content", "")).strip()
        if not msg:
            continue
        snippets.append(f"{role.upper()}: {msg}")

    merged = "\n".join(snippets)
    if len(merged) <= max_chars:
        return merged
    return "...[conversation truncated]\n" + merged[-max_chars:]


def run_diagnostic_crew(
    conversation_history: List[Dict[str, str]],
    user_message: str,
    ui_context: Dict[str, Any] | None = None,
) -> Dict[str, Any]:
    """
    Invoke the sequential Crew with explicit template inputs CrewAI substitutes into Task strings.
    """
    prime_truck_issues_index()

    ui_context = dict(ui_context or {})
    formatted_history = _format_conversation(conversation_history)
    serialized_ui = json.dumps(ui_context, indent=2, default=str)

    agents = create_all_agents()

    tasks = build_tasks(agents)

    crew = Crew(
        agents=agents,
        tasks=tasks,
        process=Process.sequential,
        verbose=False,
    )

    inputs = {
        "conversation_history": formatted_history,
        "user_message": user_message.strip(),
        "ui_context": serialized_ui,
    }

    logger.info("Starting diagnostic crew for new message")

    crew_output = crew.kickoff(inputs=inputs)

    raw = crew_output.raw

    tasks_output_texts: List[str] = []

    try:

        for to in crew_output.tasks_output:  # type: ignore[attr-defined]

            tasks_output_texts.append(getattr(to, "raw", str(to)))

    except Exception:

        pass

    return {
        "final_markdown": raw,
        "raw_output": raw,
        "tasks_output": tasks_output_texts,
    }


def warm_kb_for_streamlit() -> None:
    """
    Lightweight hook referenced from Streamlit cache layer to front-load embeddings.
    """
    prime_truck_issues_index()


In [ ]:
import os
os.makedirs("app", exist_ok=True)

In [ ]:
# --- Exported module: app/streamlit_app.py ---
# Purpose: Styled chat UX (sidebar dossier -> run_diagnostic_crew) with warmed KB caches.

%%writefile app/streamlit_app.py
"""Streamlit facade for the CrewAI-assisted truck diagnostic workflow."""

from __future__ import annotations

import html
from pathlib import Path
from typing import Any, Dict

import streamlit as st

from crewai_layer.crew_setup import run_diagnostic_crew, warm_kb_for_streamlit
from utils.config import ensure_openai_api_key
from utils.security import sanitize_user_input

_BASE = Path(__file__).resolve().parent.parent

_CUSTOM_CSS = """
<style>
.block-container {
    padding-top: 1rem;
    padding-bottom: 4rem;
    max-width: 1100px;
}
.hero {
    background: linear-gradient(118deg,#0f2027,#203a43,#2c5364);
    padding: 1.6rem 1.5rem;
    border-radius: 16px;
    color: #eef5ff;
    margin-bottom: 1.35rem;
    box-shadow: 0 15px 40px rgba(4,29,72,0.35);
}
.hero h1 { margin: 0; font-size: 1.8rem; }
.hero p {
    margin: .45rem 0 0;
    color: #cddcf5;
}
[data-testid="stSidebar"] {
    border-right: 1px solid rgba(15,63,134,0.12);
}
div[data-testid="stMarkdownContainer"] p { line-height: 1.52; }
.stSpinner > div { border-top-color:#4fc3ff !important; }
</style>
"""


def _inject_css() -> None:
    st.markdown(_CUSTOM_CSS, unsafe_allow_html=True)


@st.cache_resource(show_spinner=False)
def _bootstrap_kb() -> bool:
    warm_kb_for_streamlit()
    return True


def init_session_state() -> None:
    _load_streamlit_secret_keys()
    ensure_openai_api_key()
    if "messages" not in st.session_state:
        st.session_state["messages"] = [
            {
                "role": "assistant",
                "content": (
                    "Hi — I'm your diagnostics copilot.\n\n"
                    "Fill in the sidebar, then describe what the vehicle is actually doing "
                    "(symptoms first, fault codes optional). Always verify readings on the truck."
                ),
            },
        ]
    if "_kb_ready" not in st.session_state:
        st.session_state["_kb_ready"] = _bootstrap_kb()


def _safe_default_load(selection: list[str], value: str) -> str:
    return value if value in selection else selection[0]


def side_vehicle_form() -> Dict[str, Any]:
    load_opts = ["Empty", "Half", "Full"]
    default_load = getattr(st.session_state, "load", "Full")
    with st.sidebar:
        st.markdown("### Vehicle dossier")
        make = st.text_input("Make", value=st.session_state.get("make", ""))
        model = st.text_input("Model", value=st.session_state.get("model", ""))
        year = st.text_input("Year / series", value=st.session_state.get("year", ""))
        mileage = st.text_input("Mileage / hours", value=st.session_state.get("mileage", ""))
        ambient_temp = st.text_input("Ambient / intake temp notes", value=st.session_state.get("temp", ""))
        load_condition = st.selectbox(
            "Load Condition",
            load_opts,
            index=load_opts.index(_safe_default_load(load_opts, str(default_load))),
        )
        recent_maintenance = st.text_area("Recent Maintenance", value=st.session_state.get("recent_maint", ""))

        st.session_state.make = make.strip()
        st.session_state.model = model.strip()
        st.session_state.year = year.strip()
        st.session_state.mileage = mileage.strip()
        st.session_state.temp = ambient_temp.strip()
        st.session_state.load = load_condition
        st.session_state.recent_maint = recent_maintenance.strip()

        st.caption("Sidebar values are injected verbatim into the crew prompts.")

        return {
            "make": make.strip(),
            "model": model.strip(),
            "year": year.strip(),
            "mileage": mileage.strip(),
            "ambient_notes": ambient_temp.strip(),
            "load_condition": load_condition,
            "recent_maintenance": recent_maintenance.strip(),
        }


def _load_streamlit_secret_keys() -> None:
    for key_name in ("OPENAI_API_KEY", "OPENAI_API_KEY2", "openai_api_key2", "openai_api_key"):
        try:
            value = st.secrets.get(key_name)
        except Exception:
            continue
        if value:
            import os

            os.environ.setdefault(key_name, str(value))


def main() -> None:
    st.set_page_config(page_title="Truck Diagnostics Copilot", page_icon="🚛", layout="wide", initial_sidebar_state="expanded")

    init_session_state()
    _friendly_env_hint()
    _inject_css()

    st.markdown(
        "<div class='hero'><h1>🚛 Fleet Diagnostics Copilot</h1>"
        "<p>Grounded retrieval + procedural planning with explicit evidence separation.</p></div>",
        unsafe_allow_html=True,
    )

    ui_context = side_vehicle_form()

    for msg in st.session_state["messages"]:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    prompt = st.chat_input("What's going on with the truck right now?")
    if not prompt:
        return

    allowed, sanitized = sanitize_user_input(prompt)
    if not allowed:
        st.session_state["messages"].append({"role": "assistant", "content": sanitized})
        return

    st.session_state["messages"].append({"role": "user", "content": sanitized})
    with st.chat_message("user"):
        st.markdown(sanitized)

    with st.chat_message("assistant"):
        with st.status("Running retrieval · planner · safety review…", expanded=False) as status:
            result = run_diagnostic_crew(
                conversation_history=st.session_state["messages"],
                user_message=sanitized,
                ui_context=ui_context,
            )
            status.update(label="Complete", state="complete")
        st.markdown(result.get("final_markdown", "(no textual output captured)"))

    st.session_state["messages"].append({"role": "assistant", "content": result["final_markdown"]})


def _friendly_env_hint() -> None:
    csv_path = _BASE / "data" / "logistics_vehicle_maintenance_history.csv"
    if csv_path.exists():
        return
    st.warning(
        "Place your Kaggle-style maintenance CSV at "
        f"`{html.escape(str(csv_path))}` before expecting KB-backed answers."
    )


if __name__ == "__main__":
    main()


In [ ]:
# --- Optional QA: inspect KB columns after exports ---

from retrieval.kb_loader import load_truck_issues

df = load_truck_issues()
print("Rows:", len(df))
print("Columns:", list(df.columns)[:10])
df.head(3)


In [ ]:
import os
import stat

path = "./cloudflared-linux-amd64"

print("Exists:", os.path.exists(path))
if os.path.exists(path):
    st = os.stat(path)
    print("Mode (octal):", oct(st.st_mode))


In [ ]:
# --- Launcher: Colab-only Streamlit + tunnel helpers ---
# Local development: see README -> python -m streamlit run app/streamlit_app.py

# Cloudflare Tunnel (Dec 2025 / 2026 compatible)

!pkill -f streamlit
!pkill -f cloudflared

import os
import stat # Import the stat module for permission constants

file_path = "cloudflared-linux-amd64"

if not os.path.exists(file_path):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64

# Ensure the file is executable using os.chmod
# This is more robust than !chmod +x which sometimes doesn't persist or isn't effective
os.chmod(file_path, stat.S_IRWXU) # rwx for owner

import subprocess
import time
import sys
from IPython.display import display, HTML

# Start Streamlit
print("Starting your Truck Diagnostic Assistant...")
subprocess.Popen([
    sys.executable, "-m", "streamlit", "run", "app/streamlit_app.py",
    "--server.port=8501",
    "--server.address=0.0.0.0",
    "--server.headless=true",
    "--browser.gatherUsageStats=false"
])

time.sleep(10)

# Start Cloudflare Tunnel
print("\nStarting Cloudflare Tunnel (this takes ~10 seconds)...\n")
tunnel = subprocess.Popen(
    [f"./{file_path}", "tunnel", "--url", "http://localhost:8501"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    universal_newlines=True
)

# Robust URL extraction (works with current 2025/2026 log format)
url = None
for line in tunnel.stdout:
    line = line.strip()
    print(line)

    # Look for any line that contains a trycloudflare.com URL
    if "trycloudflare.com" in line and "https://" in line:
        # Extract the full https://...trycloudflare.com part
        start = line.find("https://")
        if start != -1:
            url = line[start:].split()[0]  # take first token after https://
            break

if url:
    display(HTML(f'''
    <div style="background:#e8f5e8; padding:25px; border-radius:15px; text-align:center; font-family:Arial; margin:20px;">
        <h1>AI Truck Diagnostic Assistant is LIVE!</h1>
        <h2 style="color:green;"><a href="{url}" target="_blank">{url}</a></h2>
        <p>Click the link above – your app is ready!</p>
    </div>
    '''))
else:
    print("Could not find tunnel URL. Check output above.")
